In [22]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils.text import extract_text
import re
from random import choice
import shutil

In [23]:
ROOT = Path("~/code/selector_examenes").expanduser()

Diccionarios de sustitución de caracteres con `str.maketrans()`

In [24]:
SUBSCRIPT = str.maketrans("0123456789+-–‒−", "₀₁₂₃₄₅₆₇₈₉₊₋₋₋₋")
SUPERSCRIPT = str.maketrans("0123456789+-–‒−", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁻⁻⁻")

In [25]:
def left_neighbor(char, chars_by_x1):
    """Carácter que termina justo donde empieza `char` (a menos de 1,5 pt) y a su misma altura."""
    x = round(char["x0"])
    candidates = [other for key in (x - 1, x, x + 1) for other in chars_by_x1.get(key, [])
                  if other is not char
                  and abs(other["x1"] - char["x0"]) < 1.5
                  and other["top"] < char["bottom"] and char["top"] < other["bottom"]]
    return min(candidates, key=lambda other: abs(other["x1"] - char["x0"]), default=None)

In [26]:
def extract_text_from_page(page):
    """Texto de la página convirtiendo los sub/superíndices en caracteres Unicode."""
    # índice de los caracteres por su borde derecho (redondeado) para encontrar vecinos rápido
    chars_by_x1 = {}
    for char in page.chars:
        chars_by_x1.setdefault(round(char["x1"]), []).append(char)

    base_of = {}       # id(índice) -> carácter normal al que acompaña (la "H" de "H₂")
    characters = []
    for char in sorted(page.chars, key=lambda char: char["x0"]):   # de izquierda a derecha
        left = left_neighbor(char, chars_by_x1)
        # si el vecino ya es un índice (el "¹" de "10⁻¹⁴"), la base es la de ese vecino
        base = base_of.get(id(left), left)
        if base is not None and char["size"] < 0.9 * base["size"]:
            # superíndice si su base queda claramente por encima de la del carácter normal
            table = SUPERSCRIPT if char["bottom"] < base["bottom"] - 2 else SUBSCRIPT
            base_of[id(char)] = base
            # se le da la altura de su base para que extract_text lo ponga en la misma línea
            char = dict(char, text=char["text"].translate(table), top=base["top"], bottom=base["bottom"])
        characters.append(char)
    # extract_text junta en una línea los caracteres cuyo top difiere como máximo y_tolerance
    # y dentro de cada línea los ordena por x0
    return extract_text(characters, y_tolerance=4)

In [27]:
def clean_page_text_2020_2024(page_text):
    """Formatea el texto del examen usando el formato de examen de 2020 - 2024"""
    text_lines = page_text.split("\n")
    # obtener índices de líneas a eliminar
    bloque_line_indexes = []
    for index, line in enumerate(text_lines):
        if "BLOQUE" in line:
            bloque_line_indexes.append(index)

    # borrar línea que empieza por BLOQUE y
    # las 3 siguientes (BLOQUE A) o
    # las 4 siguientes (BLOQUE B, C)
    line_indexes_to_remove = []
    # en cada línea que empieza por BLOQUE,
    # se marca esa línea y las 4 (ó 3) siguientes líneas
    for idx in bloque_line_indexes:
        if text_lines[idx].startswith("BLOQUE A"):
            line_indexes_to_remove.extend(list(range(idx, idx+4)))
        else:
            line_indexes_to_remove.extend(list(range(idx, idx+5)))

    # filtrar las líneas para excluir las instrucciones del bloque        
    page_text = "\n".join([text_lines[idx] for idx in range(len(text_lines))
                        if idx not in line_indexes_to_remove])

    # inserta una línea en blanco bajo cada ejercicio
    page_text = re.sub(r"(^[ABC]\d{1}\.)", r"\n\1", page_text, flags=re.MULTILINE)

    return page_text

In [28]:
def extract_exercises_from_pdf(file: Path) -> str:
    match_year = re.search(r"\d{4}", file.stem)
    if match_year:
        year = int(match_year.group(0))

    # inserto cabecera con nombre del archivo
    exam_text = f"{file.stem}\n" + "-" * len(file.stem) + "\n"

    # string donde comienzan los ejercicios
    if year <= 2019:
        start_string = "OPCIÓN"
    else:
        start_string = r"[AB]\d{1}\."

    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            page_text = extract_text_from_page(page)
            
            match = re.search(start_string, page_text)
            
            # recortar líneas iniciales hasta start_string
            if match:
                page_text = page_text[match.start():]

            # convertir a minúsculas la opción
            page_text = (page_text
                         .replace("OPCIÓN", "Opción")
                         .replace("Opción B", "\n\nOpción B")
            )

            # eliminar líneas de bloque e instrucciones
            # en exámenes 2020+
            if year >= 2020:
                page_text = clean_page_text_2020_2024(page_text)

            else:  # examen 2010 - 2019
                # inserta una línea en blanco bajo cada ejercicio
                page_text = re.sub(r"(^\d{1}\.-)", r"\n\1", page_text, flags=re.MULTILINE)
                    
            exam_text += f"{page_text}"

    return exam_text

In [29]:
file_2013 = ROOT / "examenes_quimica/2013_quimica/2013_quimica_junio_titular.pdf"
file_2022 = ROOT / "examenes_quimica/2022_quimica/2022_quimica_ordinaria_suplente.pdf"

In [30]:
junio_2013_exercises = extract_exercises_from_pdf(file_2013)
ord_suplente_2022_exercises = extract_exercises_from_pdf(file_2022)

In [ ]:
print(junio_2013_exercises)

In [ ]:
print(ord_suplente_2022_exercises)